# Die Weihnachts-Challenge: Wie viel verkauft der Onlineshop im November und Dezember?
**Grundlagen des Maschinellen Lernens | Woche 6: Zeitreihenregression | Prof. Dr. Juliana Böhmke**

**Ihr Auftrag:** Ein Online-Modehändler plant Lager, Logistik und Personal für das Weihnachtsgeschäft. Am **31.10.2020 abends** braucht er eine Prognose für den **Absatz jedes einzelnen Tages vom 01.11. bis 31.12.2020** (61 Tage). In diesen Zeitraum fallen Black Friday, die Vorweihnachtszeit und die Feiertage.

**So wird gewertet:** Mean Absolute Error (MAE) über die 61 Tage, in verkauften Artikeln pro Tag. Den wahren Absatz kennt nur die Lehrende. **Es gewinnt der niedrigste MAE.** Zur Information wird im Leaderboard zusätzlich der Root Mean Squared Error (RMSE) gezeigt; warum wir nach dem MAE entscheiden, steht in Level 2.

**Die Regeln**
1. Sie dürfen nur Informationen nutzen, die am 31.10.2020 für November und Dezember bekannt sind: Kalender, geplante Aktionen, Preise, Mediaplan, Versandplan und die Wetterprognose.
2. Der Absatz im Prognosezeitraum ist unbekannt. Was das für Ihre Merkmale bedeutet, überlegen Sie in Level 4.
3. Modelle werden nur mit den Trainingsdaten (bis 31.10.2020) verglichen und ausgewählt.
4. Abgabe: CSV-Datei mit den Spalten `datum` und `prognose`, genau 61 Zeilen. Level 7 erzeugt die Datei.

**Ablauf (ca. 120 Minuten):** Level 0 Daten ansehen, Level 1 Welche Spalten sind erlaubt?, Level 2 Validierung über mehrere Zeitfenster, Level 3 Referenzprognose, Level 4 Merkmale bauen (Ihre Entscheidung), Level 5 Modelle einstellen und vergleichen, Level 6 Feinschliff der Merkmale, Level 7 Prognose und Abgabe, Level 8 Reflexion.

Offene Aufgaben erkennen Sie an `# TODO` und `None`. Das Notebook läuft auch mit offenen Aufgaben durch und sagt Ihnen, was noch fehlt.

> Die Daten stammen von einem echten Online-Modehändler. Die absoluten Werte sind anonymisiert, Verläufe und Zusammenhänge entsprechen dem Original. Eine Beschreibung aller Spalten steht in `LIESMICH_Datenbeschreibung.md`.

## Vorbereitung
Die beiden Dateien `Onlinehandel_Absatz_Training.csv` und `Onlinehandel_Absatz_Prognosezeitraum.csv` müssen im selben Ordner liegen wie dieses Notebook. Falls XGBoost fehlt: einmalig `%pip install xgboost` ausführen und den Kernel neu starten. Diese Zelle müssen Sie nicht verändern.

In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore")
SEED = 42
STICHTAG = "2020-11-01"          # erster Tag des Prognosezeitraums
ORDNER = Path(".") if Path("Onlinehandel_Absatz_Training.csv").exists() else Path("..")

def offen(*objekte):
    # True, solange noch ein TODO offen ist.
    if any(o is None for o in objekte):
        print("Noch offen: Bitte die TODOs in der Zelle darüber ausfüllen.")
        return True
    return False

train_roh = pd.read_csv(ORDNER / "Onlinehandel_Absatz_Training.csv", parse_dates=["datum"])
zukunft_roh = pd.read_csv(ORDNER / "Onlinehandel_Absatz_Prognosezeitraum.csv", parse_dates=["datum"])
print("Training:        ", train_roh.datum.min().date(), "bis", train_roh.datum.max().date(), "|", len(train_roh), "Tage,", train_roh.shape[1], "Spalten")
print("Prognosezeitraum:", zukunft_roh.datum.min().date(), "bis", zukunft_roh.datum.max().date(), "|", len(zukunft_roh), "Tage,", zukunft_roh.shape[1], "Spalten")
train_roh.head()

## Level 0: Die Daten ansehen
Die Bedeutung aller Spalten steht in `LIESMICH_Datenbeschreibung.md`. Zielgröße ist **`absatz`**, die Anzahl verkaufter Artikel pro Tag.

**Aufgabe 0:** Führen Sie die beiden Zellen aus. Welche Muster erkennen Sie? Achten Sie auf Trend, Jahresverlauf, Wochentage und auf November und Dezember der Vorjahre.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.6))
ax.plot(train_roh.datum, train_roh.absatz, lw=0.6, color="#9AA5B1", label="täglich")
ax.plot(train_roh.datum, train_roh.absatz.rolling(28, center=True).mean(), lw=2.2, color="#003A79", label="gleitender Mittelwert (28 Tage)")
bf = train_roh[train_roh.black_friday == 1]
ax.scatter(bf.datum, bf.absatz, s=14, color="#F0883E", zorder=3, label="Black Friday")
ax.axvspan(pd.Timestamp(STICHTAG), pd.Timestamp("2020-12-31"), color="#F0883E", alpha=0.12, label="Prognosezeitraum")
ax.set_ylabel("verkaufte Artikel pro Tag"); ax.legend(frameon=False, ncol=4, loc="upper left"); plt.tight_layout(); plt.show()

print(train_roh.groupby(train_roh.datum.dt.dayofweek).absatz.mean().round(0).rename("Mittel je Wochentag (0 = Mo)").to_string())

In [ ]:
# November und Dezember der Vorjahre übereinandergelegt: Was kommt auf uns zu?
fig, ax = plt.subplots(figsize=(11, 3.4))
for jahr, farbe in ((2017, "#C9D1DA"), (2018, "#9AA5B1"), (2019, "#003A79")):
    nd = train_roh[(train_roh.datum >= f"{jahr}-10-01") & (train_roh.datum <= f"{jahr}-12-31")]
    ax.plot((nd.datum - pd.Timestamp(f"{jahr}-11-01")).dt.days, nd.absatz, color=farbe, lw=1.2, label=str(jahr))
okt = train_roh[train_roh.datum >= "2020-10-01"]
ax.plot((okt.datum - pd.Timestamp(STICHTAG)).dt.days, okt.absatz, color="#F0883E", lw=1.6, label="2020 (bis 31.10.)")
ax.axvline(0, color="black", lw=0.8, ls=":")
ax.set_xlabel("Tage ab 1. November"); ax.set_ylabel("verkaufte Artikel"); ax.legend(frameon=False, ncol=4); plt.tight_layout(); plt.show()

## Level 1: Welche Spalten dürfen wir nutzen?
Die Prognose entsteht am 31.10.2020. Eine Spalte darf nur in das Modell, wenn ihr Wert für November und Dezember zu diesem Zeitpunkt schon bekannt ist. Sonst nutzt das Modell Informationen aus der Zukunft, das nennt man Data Leakage.

Die nächste Zelle zeigt, welche Spalten es nur im Training gibt.

In [ ]:
nur_training = [c for c in train_roh.columns if c not in zukunft_roh.columns]
print("Nur in den Trainingsdaten:", nur_training)

**Aufgabe 1:** Tragen Sie die Liste der erlaubten Merkmale ein. `datum` und die Zielgröße `absatz` gehören nicht dazu (die Kalendermerkmale bauen wir in Level 4 aus dem Datum).

In [ ]:
# TODO: Liste der erlaubten Spalten, z. B. ["feiertag", "black_friday", ...]
ERLAUBT = None

if not offen(ERLAUBT):
    verboten = [c for c in ERLAUBT if c not in zukunft_roh.columns or c in ("datum", "absatz")]
    print("Erlaubt:", len(ERLAUBT), "Spalten", "| Achtung, nicht erlaubt:" if verboten else "", verboten if verboten else "")

Die folgende Zelle zeigt, was passiert, wenn man `umsatz` doch nutzt: eine lineare Regression auf den Logarithmus von Umsatz und Durchschnittspreis, Training bis 31.08.2020, Prüfung September und Oktober 2020. Führen Sie sie aus.

In [ ]:
a, b = train_roh[train_roh.datum < "2020-09-01"], train_roh[(train_roh.datum >= "2020-09-01") & (train_roh.datum < STICHTAG)]
spalten = ["umsatz", "durchschnittspreis"]
lm = LinearRegression().fit(np.log(a[spalten]), np.log(a.absatz))
print(f"MAE mit {spalten}: {mean_absolute_error(b.absatz, np.exp(lm.predict(np.log(b[spalten])))):,.0f}")
print("Zum Vergleich: Absatz im Mittel", f"{b.absatz.mean():,.0f}", "Artikel pro Tag; die besten erlaubten Modelle liegen in Level 5 bei 800 bis 1.200.")

**Kurzfrage 1:** Warum ist das Ergebnis mit `umsatz` zu schön, um wahr zu sein? Warum steht `lockdown_einzelhandel` nicht im Prognosezeitraum, obwohl es ab dem 16.12.2020 einen Lockdown gab?

## Level 2: Validierung über mehrere Zeitfenster
Wir prognostizieren 61 Tage auf einmal. Also prüfen wir Modelle genauso: Training bis zu einem Tag, dann **zwei Monate am Stück** vorhersagen, ohne den Absatz dieser zwei Monate zu kennen. Ein einziges Zeitfenster kann täuschen (Level 5 zeigt, wie stark die Ränge schwanken). Deshalb nutzen wir drei Fenster hintereinander und mitteln: rollierende Validierung / Rolling-Origin-Validierung.

```
Training ─────────────────────────────┤ Mai/Jun │
Training ───────────────────────────────────────┤ Jul/Aug │
Training ─────────────────────────────────────────────────┤ Sep/Okt │
Training ───────────────────────────────────────────────────────────┤ Nov/Dez = Abgabe │
```

**Aufgabe 2:** Ergänzen Sie die Fenster Juli/August und September/Oktober 2020 (Start und Ende jeweils als Text `"JJJJ-MM-TT"`).

**Welches Fehlermaß?** `validiere` berichtet zwei Maße:
- **MAE** (Mean Absolute Error) je Fenster und im Mittel: im Mittel so viele Artikel pro Tag daneben. Danach wählen wir aus, weil auch die Abgabe nach dem MAE gewertet wird.
- **RMSE** (Root Mean Squared Error), zur Information: Große Fehler zählen überproportional. Den MSE selbst berichten wir nicht, er liegt in „Artikel²“ und ist kaum zu lesen. RMSE und MSE ergeben immer dieselbe Rangfolge.

Ein Modell, das beim MAE vorne liegt, beim RMSE aber zurückfällt, hat einzelne große Fehltage, zum Beispiel am Black Friday.

In [ ]:
FENSTER = {
    "Mai/Jun": ("2020-05-01", "2020-06-30"),
    # TODO: "Jul/Aug": (..., ...),
    # TODO: "Sep/Okt": (..., ...),
}

def validiere(prognose_fn, daten, **kw):
    # prognose_fn(train, ziel, **kw) bekommt nur Tage VOR dem Fenster mit Absatz; im Fenster darf es den Absatz nicht benutzen.
    # Ergebnis: MAE je Fenster, "Mittel" = mittlerer MAE (entscheidet), "RMSE" = mittlerer RMSE (zur Information)
    ergebnis, rmse = {}, []
    for name, (start, ende) in FENSTER.items():
        tr = daten[daten.datum < start]
        zi = daten[(daten.datum >= start) & (daten.datum <= ende)]
        p = prognose_fn(tr, zi.drop(columns="absatz"), **kw)
        ergebnis[name] = mean_absolute_error(zi.absatz, p)
        rmse.append(np.sqrt(mean_squared_error(zi.absatz, p)))
    ergebnis["Mittel"] = np.mean(list(ergebnis.values()))
    ergebnis["RMSE"] = np.mean(rmse)
    return ergebnis

if len(FENSTER) < 3:
    print("Noch offen: Bitte die beiden fehlenden Fenster eintragen.")
else:
    for name, (start, ende) in FENSTER.items():
        print(f"{name}: Training bis {pd.Timestamp(start) - pd.Timedelta(days=1):%d.%m.%Y}, Prüfung {start} bis {ende} ({(pd.Timestamp(ende) - pd.Timestamp(start)).days + 1} Tage)")

**Kurzfrage 2:** Warum nutzen wir hier nicht `TimeSeriesSplit` mit fünf Folds über die ganzen Trainingsdaten, und warum keine Fenster von einer Woche?

**Kurzfrage 2b:** Der Händler fragt, warum nach dem MAE gewertet wird und nicht nach dem MSE. Was antworten Sie? Wann wäre der RMSE die bessere Wahl?

## Level 3: Die Referenzprognose
Die einfachste sinnvolle Prognose für zwei Monate: **Derselbe Wochentag vor 52 Wochen** (364 Tage zurück), die saisonal naive Prognose. Sie kennt den Jahresverlauf, Black Friday und Weihnachten des Vorjahres, aber nicht das Wachstum. Jedes Modell muss diesen Referenzwert unterbieten.

**Aufgabe 3:** Ergänzen Sie die Funktion. Tipp: `reihe.reindex(...)` holt die Werte zu einer Liste von Daten.

In [ ]:
def saisonal_naiv(tr, zi):
    reihe = tr.set_index("datum").absatz
    # TODO: Daten 364 Tage vor den Zieltagen bilden und die Werte aus reihe holen
    vorjahr = None
    if vorjahr is None:
        return np.full(len(zi), np.nan)
    return reihe.reindex(vorjahr).values

if FENSTER is not None and len(FENSTER) >= 3:
    ref = validiere(saisonal_naiv, train_roh)
    if not np.isnan(ref["Mittel"]):
        print({k: round(v) for k, v in ref.items()})

**Kurzfrage 3:** Warum 364 und nicht 365 Tage? Was wird die saisonal naive Prognose im Prognosezeitraum vermutlich falsch machen?

## Level 4: Merkmale / Features bauen – Ihre Entscheidung
Ab hier entscheiden Sie selbst. Die Modelle kennen keine Zeit: Trend, Wochenmuster, Jahresverlauf und Sonderereignisse erfahren sie nur über Merkmale, die **Sie** bauen. Ideen und Code finden Sie in der Vorlesung und im Notebook *Codebeispiele UrbanBike*.

Fragen, die Ihnen helfen:
- Wie bekommt ein Modell den **Trend** mit? Funktioniert das für jedes Modell gleich?
- Wie stellen Sie **Wochenmuster** und **Jahresverlauf** dar (Zahl, Dummies, sin/cos)? Welche Form passt zu welchem Modell?
- Brauchen **Sonderereignisse** (Black Friday, Rabatte, Feiertage, Weihnachten) eine eigene Darstellung?
- Wirken **Werbebudgets** linear, oder bringen die ersten 1.000 € mehr als die nächsten?
- Welche **Lags** des Absatzes sind bei 61 Tagen Horizont erlaubt?

Die Funktion `merkmale(df)` bekommt Training und Prognosezeitraum zusammen (im Prognosezeitraum ist `absatz` leer) und gibt sie mit Ihren zusätzlichen Spalten zurück. So entstehen die Merkmale für beide Teile auf dieselbe Weise.

**Aufgabe 4:** Bauen Sie Ihre Merkmale. Sie dürfen bestehende Spalten auch umwandeln.

In [ ]:
def merkmale(df):
    df = df.copy()
    # TODO: Ihre Merkmale, z. B. df["mein_merkmal"] = ...

    return df

alle = merkmale(pd.concat([train_roh, zukunft_roh], ignore_index=True))
daten = alle[alle.datum < STICHTAG].copy()                     # Training mit Absatz
zukunft = alle[alle.datum >= STICHTAG].copy()                  # Prognosezeitraum ohne Absatz

# Prüfzelle: Welche Spalten sind im Prognosezeitraum vollständig und damit verwendbar?
gesperrt = ["datum", "absatz"] + nur_training
mit_missing_values = [c for c in alle.columns if c not in gesperrt and zukunft[c].isna().any()]
VERFUEGBAR = [c for c in alle.columns if c not in gesperrt and c not in mit_missing_values]
print("Neue Spalten:", [c for c in alle.columns if c not in train_roh.columns] or "noch keine")
if mit_missing_values:
    print("Achtung, Missing Values im Prognosezeitraum, deshalb NICHT verwendbar:", mit_missing_values)
print(f"Verwendbar ({len(VERFUEGBAR)}):", VERFUEGBAR)

**Kurzfrage 4:** Begründen Sie Ihre Merkmale in zwei, drei Sätzen. Warum würde `lag_1` (Absatz von gestern) in der Prüfzelle als unvollständig gemeldet?

## Level 5: Modelle einstellen und vergleichen
Fünf Modelle stehen bereit. Jedes ist eine Funktion `modell(tr, zi, merkmale, log_ziel, ...)`: Sie trainiert auf `tr` mit genau den Merkmalen, die Sie angeben, und gibt für jeden Tag in `zi` eine Prognose zurück. Mit `log_ziel=True` lernt das Modell $\log(\text{absatz})$ statt `absatz`; die Prognose wird automatisch zurückgerechnet. Trainingszeilen mit Missing Values in den Merkmalen (z. B. durch lange Lags) werden ausgelassen.

| Modell | Besonderheit |
|---|---|
| Lineare Regression | schreibt Zusammenhänge linear fort, auch über das Trainingsniveau hinaus |
| Random Forest, XGBoost | nichtlinear, Wechselwirkungen; Prognosen bleiben im Bereich der Trainingswerte |
| Trend + XGBoost | Trendbereinigung: Gerade auf der Spalte `t`, XGBoost lernt den Rest mit den übrigen Merkmalen. Braucht eine Spalte `t`. |
| SARIMAX | Wochensaison über `saison`, Merkmale als exogene Variablen |

In [ ]:
def rf_modell():
    return RandomForestRegressor(n_estimators=400, min_samples_leaf=3, random_state=SEED, n_jobs=-1)

def xgb_modell():
    return XGBRegressor(n_estimators=600, learning_rate=0.03, max_depth=4, subsample=0.8, colsample_bytree=0.8, random_state=SEED)

def _ziel(tr, log_ziel):
    return np.log(tr.absatz) if log_ziel else tr.absatz

def _zurueck(p, log_ziel):
    return np.exp(p) if log_ziel else np.asarray(p)

def lineare_regression(tr, zi, merkmale, log_ziel=True):
    tr = tr.dropna(subset=merkmale)
    m = LinearRegression().fit(tr[merkmale], _ziel(tr, log_ziel))
    return _zurueck(m.predict(zi[merkmale]), log_ziel)

def random_forest(tr, zi, merkmale, log_ziel=False):
    tr = tr.dropna(subset=merkmale)
    return _zurueck(rf_modell().fit(tr[merkmale], _ziel(tr, log_ziel)).predict(zi[merkmale]), log_ziel)

def xgboost(tr, zi, merkmale, log_ziel=False):
    tr = tr.dropna(subset=merkmale)
    return _zurueck(xgb_modell().fit(tr[merkmale], _ziel(tr, log_ziel)).predict(zi[merkmale]), log_ziel)

def trend_xgboost(tr, zi, merkmale, log_ziel=True):
    tr = tr.dropna(subset=merkmale)
    y = _ziel(tr, log_ziel)
    trend = LinearRegression().fit(tr[["t"]], y)                       # Schritt 1: Trend als Gerade
    rest = [m for m in merkmale if m != "t"]
    baum = xgb_modell().fit(tr[rest], y - trend.predict(tr[["t"]]))   # Schritt 2: XGBoost lernt den Rest
    return _zurueck(trend.predict(zi[["t"]]) + baum.predict(zi[rest]), log_ziel)

def sarimax(tr, zi, merkmale, log_ziel=True, ordnung=(1, 1, 1), saison=(1, 0, 1, 7)):
    tr = tr.dropna(subset=merkmale)
    X = list(merkmale) or None
    m = SARIMAX(_ziel(tr, log_ziel).values, exog=tr[X].values if X else None, order=ordnung, seasonal_order=saison).fit(disp=False)
    return _zurueck(m.forecast(len(zi), exog=zi[X].values if X else None), log_ziel)

MODELLE = {"Lineare Regression": lineare_regression, "Random Forest": random_forest, "XGBoost": xgboost,
           "Trend + XGBoost": trend_xgboost, "SARIMAX": sarimax}
print("Modelle bereit:", list(MODELLE))

**Aufgabe 5:** Tragen Sie für jedes Modell Ihre Merkmale ein und entscheiden Sie über `log_ziel` und die übrigen Einstellungen. Die Werte, die schon dastehen, sind nur Startwerte. Ein Modell, dessen `merkmale` noch `None` sind, wird übersprungen. Sie müssen also nicht alle fünf Modelle nutzen.

Beachten Sie: Die Modelle bekommen **nur** die Spalten, die Sie angeben. Kalender und Trend kommen nicht automatisch dazu (außer bei SARIMAX, das Trend und Wochensaison selbst modelliert).

In [ ]:
EINSTELLUNGEN = {
    "Lineare Regression": dict(merkmale=None, log_ziel=True),        # TODO
    "Random Forest":      dict(merkmale=None, log_ziel=False),       # TODO
    "XGBoost":            dict(merkmale=None, log_ziel=False),       # TODO
    "Trend + XGBoost":    dict(merkmale=None, log_ziel=True),        # TODO (braucht die Spalte "t")
    "SARIMAX":            dict(merkmale=None, log_ziel=True, ordnung=(1, 1, 1), saison=(1, 0, 1, 7)),  # TODO
}

Jetzt der Vergleich über die drei Validierungsfenster aus Level 2. Das dauert etwa eine Minute (SARIMAX). Sie müssen nichts ändern.

Die Spalten: MAE je Fenster, `Mittel` = mittlerer MAE (danach ist sortiert, danach wählen Sie aus), `RMSE` = mittlerer RMSE zur Information. Achten Sie darauf, ob sich die Rangfolge beim RMSE ändert.

In [ ]:
def pruefe(name, cfg):
    mm = cfg["merkmale"]
    if mm is None:
        return False
    falsch = [m for m in mm if m not in VERFUEGBAR]
    if falsch:
        print(f"{name}: nicht verwendbar {falsch} (fehlen, unvollständig im Prognosezeitraum oder nur im Training). Übersprungen.")
        return False
    return True

if len(FENSTER) >= 3:
    zeilen = {"Saisonal naiv (vor 52 Wochen)": validiere(saisonal_naiv, daten)}
    for name, cfg in EINSTELLUNGEN.items():
        if name in MODELLE and pruefe(name, cfg):
            try:
                zeilen[name] = validiere(MODELLE[name], daten, **cfg)
            except Exception as e:
                print(f"{name}: Fehler ({type(e).__name__}: {str(e)[:80]}). Bitte die Einstellungen prüfen.")
    vergleich = pd.DataFrame(zeilen).T.sort_values("Mittel")
    display(vergleich.round(0).astype(int))
    if len(vergleich) == 1:
        print("Noch offen: Bitte in EINSTELLUNGEN Merkmale eintragen.")

Die Prognosen der beiden besten Modelle im letzten Fenster, September und Oktober 2020:

In [ ]:
beste = [z for z in vergleich.index if "naiv" not in z][:2] if "vergleich" in globals() else []
if beste:
    start, ende = FENSTER["Sep/Okt"]
    tr, zi = daten[daten.datum < start], daten[(daten.datum >= start) & (daten.datum <= ende)]
    plt.figure(figsize=(11, 3.4))
    plt.plot(zi.datum, zi.absatz, color="#9AA5B1", lw=1.4, label="wahr")
    for name, farbe in zip(beste, ("#003A79", "#F0883E")):
        plt.plot(zi.datum, MODELLE[name](tr, zi.drop(columns="absatz"), **EINSTELLUNGEN[name]), color=farbe, lw=1.4, label=name)
    plt.legend(frameon=False); plt.ylabel("verkaufte Artikel"); plt.tight_layout(); plt.show()

**Kurzfrage 5:**
1. Welches Modell gewinnt im Mittel? Wie stark schwanken die Ränge zwischen den drei Fenstern?
2. Sollte ein Baum (Random Forest, XGBoost) ein Trendmerkmal `t` bekommen? Probieren Sie XGBoost mit und ohne `t` und vergleichen Sie mit Trend + XGBoost. Was passiert mit `t` in einem Baum bei Tagen nach dem Trainingsende?
3. Warum ist SARIMAX bei 61 Tagen Horizont schwächer als bei einem Tag Horizont (UrbanBike)?

## Level 6: Feinschliff – welche Merkmale helfen wirklich?
Nicht jede Spalte hilft. Mit `hinzunahme_test` prüfen Sie für ein Modell, wie sich der mittlere Validierungs-MAE ändert, wenn **eine** Spalte zu Ihren Merkmalen hinzukommt (negativ = besser). Mit `weglass_test` prüfen Sie umgekehrt, was passiert, wenn eine Ihrer Spalten wegfällt (positiv = die Spalte war nützlich).

Tipps:
- Die lineare Regression rechnet in Sekunden, SARIMAX braucht pro Spalte etwa 10 bis 20 Sekunden.
- Eine Spalte, die einzeln kaum hilft, kann zusammen mit anderen nützen, und umgekehrt.
- Keines der drei Validierungsfenster enthält einen Black Friday. Überlegen Sie fachlich, welche Spalten im November und Dezember wichtig werden, auch wenn der Test sie nicht belohnen kann.

**Aufgabe 6:** Testen Sie Ihre Merkmale und passen Sie `EINSTELLUNGEN` in Level 5 an. Führen Sie danach den Vergleich erneut aus.

In [ ]:
def hinzunahme_test(name, kandidaten):
    cfg = EINSTELLUNGEN[name]
    basis = validiere(MODELLE[name], daten, **cfg)["Mittel"]
    eff = {c: validiere(MODELLE[name], daten, **dict(cfg, merkmale=cfg["merkmale"] + [c]))["Mittel"] - basis
           for c in kandidaten if c not in cfg["merkmale"]}
    return pd.Series(eff).sort_values().round(0)

def weglass_test(name):
    cfg = EINSTELLUNGEN[name]
    basis = validiere(MODELLE[name], daten, **cfg)["Mittel"]
    eff = {c: validiere(MODELLE[name], daten, **dict(cfg, merkmale=[m for m in cfg["merkmale"] if m != c]))["Mittel"] - basis
           for c in cfg["merkmale"]}
    return pd.Series(eff).sort_values(ascending=False).round(0)

In [ ]:
# Beispiel (ausprobieren und anpassen):
# hinzunahme_test("Lineare Regression", VERFUEGBAR)
# weglass_test("Lineare Regression")

**Kurzfrage 6:** Welche Spalten haben wenig Aussagekraft? Warum ist „alle Spalten nehmen“ nicht automatisch am besten?

**Bonus:** Ein Ensemble mittelt die Prognosen zweier Modelle. Vorlage:
```python
def ensemble(tr, zi, **kw):
    return (lineare_regression(tr, zi, **EINSTELLUNGEN["Lineare Regression"]) + xgboost(tr, zi, **EINSTELLUNGEN["XGBoost"])) / 2
MODELLE["Ensemble"] = ensemble; EINSTELLUNGEN["Ensemble"] = dict(merkmale=[])
print(validiere(ensemble, daten))
```

## Level 7: Prognose und Abgabe
Wählen Sie Ihr Modell. Die Zelle trainiert es mit Ihren Einstellungen auf **allen** Trainingsdaten bis 31.10.2020, sagt die 61 Tage vorher und schreibt die Abgabedatei.

**Aufgabe 7:** Tragen Sie Teamnamen und Modell ein (Name wie in `MODELLE`, z. B. `"Lineare Regression"`).

In [ ]:
TEAM = None          # TODO: Teamname ohne Leerzeichen, z. B. "Team_Prognose"
SIEGER = None        # TODO: Modellname aus MODELLE

if not offen(TEAM, SIEGER):
    prognose = np.asarray(MODELLE[SIEGER](daten, zukunft.drop(columns="absatz"), **EINSTELLUNGEN[SIEGER]))
    abgabe = pd.DataFrame({"datum": zukunft.datum.dt.strftime("%Y-%m-%d").values, "prognose": prognose.round(0).astype(int)})
    assert len(abgabe) == 61 and abgabe.prognose.notna().all()
    datei = f"Abgabe_{TEAM}.csv"
    abgabe.to_csv(datei, index=False)
    print(f"{datei} gespeichert: {len(abgabe)} Zeilen, Summe {abgabe.prognose.sum():,} Artikel, Mittel {abgabe.prognose.mean():,.0f} pro Tag")

    vj = daten.set_index("datum").absatz
    plt.figure(figsize=(11, 3.4))
    plt.plot(daten.datum[daten.datum >= "2020-08-01"], daten.absatz[daten.datum >= "2020-08-01"], color="#9AA5B1", lw=1, label="Absatz bis 31.10.2020")
    plt.plot(zukunft.datum, vj.reindex(zukunft.datum - pd.Timedelta(days=364)).values, color="#C9D1DA", lw=1, ls="--", label="Vorjahr (vor 52 Wochen)")
    plt.plot(zukunft.datum, prognose, color="#003A79", lw=1.8, label=f"Prognose {SIEGER}")
    plt.legend(frameon=False); plt.ylabel("verkaufte Artikel"); plt.tight_layout(); plt.show()

**Abgabe:** Laden Sie die Datei `Abgabe_<Team>.csv` hoch und notieren Sie Modell, Merkmale und Ihren Validierungs-MAE.

## Level 8: Reflexion
1. Warum darf `budget_suchmaschine` ins Modell, `google_suchinteresse` aber nicht, obwohl beides mit Suchen zu tun hat?
2. Warum haben wir am Ende auf allen Trainingsdaten neu trainiert und nicht das Modell aus dem Fenster Sep/Okt verwendet?
3. Was passiert mit Ihrer Prognose, wenn der Händler im Dezember das Suchmaschinenbudget kurzfristig verdoppelt? Ist das eine Stärke oder ein Risiko des Modells?
4. Im echten Dezember 2020 wurde ab dem 16.12. der stationäre Einzelhandel geschlossen. In welche Richtung wird Ihre Prognose daneben liegen? Hätte ein Modell das wissen können?
5. Welches Ihrer selbst gebauten Merkmale hat am meisten gebracht, und warum?